# OLYMPIC AI 2026 - TÁC VỤ 2: PHÂN LOẠI RÁC THẢI (TRASHNET 6 CLASSES)
## Notebook: Quy trình Huấn luyện & Sinh file kết quả dự đoán (generate_result.ipynb)

Notebook này chứa toàn bộ pipeline từ tiền xử lý dữ liệu, định nghĩa kiến trúc mô hình, huấn luyện/fine-tuning (sử dụng **EfficientNet-B2**), đánh giá trên tập Validation với chỉ số **Macro F1-Score**, và sinh file kết quả dự đoán `submission.csv`.

* **Kiến trúc mô hình**: `efficientnet_b2` (timm pretrained trên ImageNet-1k)
* **Số lớp phân loại**: 6 classes (1: cardboard, 2: glass, 3: metal, 4: paper, 5: plastic, 6: trash)
* **Metric mục tiêu**: Macro F1-Score & Điểm chuẩn hóa BTC
* **Output**: `submission.csv` với 2 cột `file_name,category_id`


In [ ]:
# 1. Khai báo các thư viện cần thiết
import os
import sys
import random
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import timm

print(f"PyTorch Version: {torch.__version__}")
print(f"Timm Version: {timm.__version__}")


In [ ]:
# 2. Thiết lập Seed tái lập kết quả & Tự động phát hiện đường dẫn dữ liệu
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

def resolve_paths():
    candidates = [
        ".",
        "./data/TACVU2",
        "../data/TACVU2",
        os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else ".",
        os.path.join(os.path.dirname(os.path.abspath("__file__")), "data/TACVU2") if "__file__" in globals() else "."
    ]
    train_dir = None
    root_dir = "."
    for c in candidates:
        if os.path.exists(os.path.join(c, "train", "images")) and os.path.exists(os.path.join(c, "train", "train.csv")):
            train_dir = os.path.join(c, "train")
            root_dir = c
            break
        elif os.path.exists(os.path.join(c, "images")) and os.path.exists(os.path.join(c, "train.csv")):
            train_dir = c
            root_dir = os.path.dirname(c) if os.path.dirname(c) else "."
            break
            
    if train_dir is None:
        train_dir = "./train"
        root_dir = "."

    # Tự động giải nén private_test.zip nếu có mật khẩu / giải nén được
    priv_zip = os.path.join(root_dir, "private_test.zip")
    priv_dir = os.path.join(root_dir, "private_test")
    if os.path.exists(priv_zip) and (not os.path.exists(os.path.join(priv_dir, "images")) or len(os.listdir(os.path.join(priv_dir, "images"))) == 0):
        for pwd in [b'ckolp26@vku', None]:
            try:
                with zipfile.ZipFile(priv_zip, "r") as z:
                    z.extractall(root_dir, pwd=pwd)
                break
            except Exception:
                pass

    test_dir = None
    for cand in [
        os.path.join(root_dir, "private_test"),
        os.path.join(root_dir, "public_test"),
        "./private_test",
        "./public_test",
        "./test"
    ]:
        img_p = os.path.join(cand, "images")
        if os.path.exists(img_p) and len([f for f in os.listdir(img_p) if f.endswith(('.jpg', '.jpeg', '.png'))]) > 0:
            test_dir = cand
            break
            
    if test_dir is None:
        test_dir = os.path.join(root_dir, "public_test")

    return os.path.abspath(root_dir), os.path.abspath(train_dir), os.path.abspath(test_dir)

ROOT_DIR, TRAIN_DIR, TEST_DIR = resolve_paths()
print(f"ROOT_DIR : {ROOT_DIR}")
print(f"TRAIN_DIR: {TRAIN_DIR}")
print(f"TEST_DIR : {TEST_DIR}")


In [ ]:
# 3. Data Augmentation & Định nghĩa Dataset
# Sử dụng Augmentation hợp lý nhằm chống overfit nhưng không làm mờ đặc trưng vật liệu
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

class TrashDataset(Dataset):
    """
    Dataset cho bài toán phân loại rác thải 6 lớp (TrashNet)
    - Labels gốc trong CSV: 1-6 (cardboard, glass, metal, paper, plastic, trash)
    - PyTorch Labels: 0-5
    """
    def __init__(self, root_dir, df=None, csv_file_name=None, transform=None, is_test=False):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.transform = transform
        self.is_test = is_test

        if is_test:
            self.image_files = sorted([f for f in os.listdir(self.img_dir)
                                      if f.endswith(('.jpg', '.jpeg', '.png'))])
            self.annotations = pd.DataFrame({'file_name': self.image_files})
        elif df is not None:
            self.annotations = df.reset_index(drop=True)
        else:
            self.annotations = pd.read_csv(os.path.join(root_dir, csv_file_name))

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        img_path = os.path.join(self.img_dir, row['file_name'])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, -1

        label = int(row['category_id']) - 1
        return image, label


In [ ]:
# 4. Phân chia Train/Validation có bảo toàn tỷ lệ nhãn (Stratified Split) & DataLoaders
BATCH_SIZE = 32
VAL_SPLIT = 0.2

df_full = pd.read_csv(os.path.join(TRAIN_DIR, "train.csv"))
train_df, val_df = train_test_split(
    df_full, 
    test_size=VAL_SPLIT, 
    random_state=SEED, 
    stratify=df_full['category_id']
)

print(f"Tổng số mẫu: {len(df_full)}")
print(f"Số mẫu Train: {len(train_df)}")
print(f"Số mẫu Validation: {len(val_df)}")

train_dataset = TrashDataset(root_dir=TRAIN_DIR, df=train_df, transform=train_transforms)
val_dataset = TrashDataset(root_dir=TRAIN_DIR, df=val_df, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


In [ ]:
# 5. Khởi tạo mô hình EfficientNet-B2 và Cấu hình Huấn luyện với Focal Loss
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Thiết bị huấn luyện (Device): {device}")

# Định nghĩa Focal Loss giúp mô hình tập trung vào hard examples và bù trừ class imbalance
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss

        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Tính toán alpha (trọng số lớp) dựa trên nghịch đảo tần suất từ train_df
class_counts = train_df['category_id'].value_counts().sort_index().values
class_counts = torch.tensor(class_counts, dtype=torch.float32)
weights = 1.0 / (class_counts ** 0.5)
alpha = (weights / weights.sum()).to(device)
print(f"Phân phối số mẫu train: {class_counts.tolist()}")
print(f"Trọng số alpha tương ứng: {[round(x, 4) for x in alpha.tolist()]}")

# Mô hình EfficientNet-B2 với mức regularization vừa vặn (drop_rate=0.1, drop_path_rate=0.0) tránh underfitting
model = timm.create_model(
    'efficientnet_b2', 
    pretrained=True, 
    num_classes=6,
    drop_rate=0.1,
    drop_path_rate=0.0
)
model.to(device)

criterion = FocalLoss(alpha=alpha, gamma=1.5)

# Optimizer AdamW với learning rate 2e-4 và weight_decay 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)

NUM_EPOCH = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCH, eta_min=1e-6)

print(f"Tổng số tham số mô hình: {sum(p.numel() for p in model.parameters()):,}")
print("✅ Đã khởi tạo Focal Loss (gamma=1.5, alpha class-weights) thành công!")


In [ ]:
# 6. Vòng lặp Huấn luyện & Đánh giá (Tracking Macro F1 & Lưu Best Model)
best_model_path = os.path.join(ROOT_DIR, "best_model.pth")
best_macro_f1 = 0.0
MIN_SCORE = 0.0
MAX_SCORE = 0.9543

history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_macro_f1": []
}

print(f"Bắt đầu huấn luyện mô hình trong {NUM_EPOCH} epochs...")

for epoch in range(NUM_EPOCH):
    # ===== TRAINING PHASE =====
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm.tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCH} [Train]"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # ===== VALIDATION PHASE =====
    model.eval()
    val_running_loss = 0.0
    all_predicts = []
    all_targets = []

    with torch.no_grad():
        for images, labels in tqdm.tqdm(val_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCH} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            all_predicts.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    val_loss = val_running_loss / len(val_dataset)
    val_acc = 100.0 * (np.array(all_predicts) == np.array(all_targets)).mean()
    val_macro_f1 = f1_score(all_targets, all_predicts, average="macro")

    if val_macro_f1 > MIN_SCORE:
        final_score = 100.0 * (val_macro_f1 - MIN_SCORE) / (MAX_SCORE - MIN_SCORE)
    else:
        final_score = 0.0

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # Lưu lịch sử
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)

    # Lưu Best Checkpoint dựa trên Macro F1
    best_msg = ""
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        torch.save(model.state_dict(), best_model_path)
        best_msg = f"  ===> [LƯU BEST MODEL] (F1: {val_macro_f1*100:.2f}% | Điểm BTC: {final_score:.2f})"

    print(f"Epoch [{epoch + 1:02d}/{NUM_EPOCH}]:")
    print(f"  Train: Loss = {train_loss:.4f}, Acc = {train_acc:.2f}%")
    print(f"  Val  : Loss = {val_loss:.4f}, Acc = {val_acc:.2f}%, Macro F1 = {val_macro_f1*100:.2f}% (Điểm: {final_score:.2f})")
    print(f"  LR: {current_lr:.6f}{best_msg}")
    print("-" * 65)

print(f"Hoàn thành huấn luyện! Kỷ lục Macro F1 cao nhất: {best_macro_f1*100:.2f}%")


In [ ]:
# 7. Trực quan hóa đường cong học tập (Loss, Accuracy & Macro F1 Curves)
epochs = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(18, 5))

# 1. Loss Curve
plt.subplot(1, 3, 1)
plt.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
plt.plot(epochs, history['val_loss'], 'r--s', label='Val Loss', linewidth=2)
plt.title('Loss Curve (Train vs Validation)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# 2. Accuracy Curve
plt.subplot(1, 3, 2)
plt.plot(epochs, history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
plt.plot(epochs, history['val_acc'], 'r--s', label='Val Acc', linewidth=2)
plt.title('Accuracy Curve (%)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# 3. Macro F1 Curve
plt.subplot(1, 3, 3)
f1_pct = [f * 100 for f in history['val_macro_f1']]
plt.plot(epochs, f1_pct, 'g-^', label='Val Macro F1', linewidth=2)
if len(f1_pct) > 0:
    best_idx = int(np.argmax(history['val_macro_f1']))
    plt.plot(epochs[best_idx], f1_pct[best_idx], 'r*', markersize=14, label=f'Best F1: {f1_pct[best_idx]:.2f}%')
plt.title('Validation Macro F1 (%)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Macro F1 (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# 8. Tối ưu ngưỡng hậu xử lý (Threshold Tuning) & Dự đoán trên Test Set

# 1. Tải lại Best Checkpoint vừa huấn luyện
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print(f"Đã nạp trọng số tốt nhất từ {best_model_path}")
else:
    print("Sử dụng trọng số hiện tại của mô hình.")

model.eval()

# 2. Trích xuất xác suất dự đoán trên tập Validation để tối ưu ngưỡng
print("Đang đánh giá xác suất trên tập Validation...")
val_probs, val_targets = [], []
with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        val_probs.append(probs)
        val_targets.append(targets.numpy())

val_probs = np.vstack(val_probs)
val_targets = np.concatenate(val_targets)

# 3. Thuật toán tìm bộ trọng số Multipliers tối ưu hóa Macro F1
def optimize_class_multipliers(probs, targets, num_classes=6):
    best_w = np.ones(num_classes)
    best_f1 = f1_score(targets, np.argmax(probs, axis=1), average='macro')
    
    for it in range(5):
        improved = False
        for c in range(num_classes):
            for cand in np.linspace(0.5, 2.5, 41):
                tw = best_w.copy()
                tw[c] = cand
                score = f1_score(targets, np.argmax(probs * tw, axis=1), average='macro')
                if score > best_f1 + 1e-4:
                    best_f1 = score
                    best_w = tw.copy()
                    improved = True
        if not improved:
            break
    return best_w, best_f1

base_val_f1 = f1_score(val_targets, np.argmax(val_probs, axis=1), average='macro')
best_multipliers, tuned_val_f1 = optimize_class_multipliers(val_probs, val_targets)
print(f"Macro F1 gốc (Argmax): {base_val_f1*100:.2f}%")
print(f"Macro F1 sau Threshold Tuning: {tuned_val_f1*100:.2f}% (Tăng: +{(tuned_val_f1 - base_val_f1)*100:.2f}%)")
print(f"Bộ trọng số tối ưu (w): {[round(x, 4) for x in best_multipliers.tolist()]}")

# 4. Tạo DataLoader cho Test Set
test_dataset = TrashDataset(
    root_dir=TEST_DIR,
    transform=val_transforms,
    is_test=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"\nBắt đầu Inference trên {len(test_dataset)} ảnh thuộc: {TEST_DIR}...")
file_names = []
predictions = []

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm.tqdm(test_loader, desc="Inference")):
        images = images.to(device)
        
        # Test-Time Augmentation (TTA): ảnh gốc + ảnh lật ngang
        outputs_orig = model(images)
        outputs_flip = model(torch.flip(images, dims=[3]))
        
        probs_orig = torch.softmax(outputs_orig, dim=1).cpu().numpy()
        probs_flip = torch.softmax(outputs_flip, dim=1).cpu().numpy()
        avg_probs = (probs_orig + probs_flip) / 2.0
        
        # Áp dụng bộ trọng số tối ưu hóa Macro F1
        scaled_probs = avg_probs * best_multipliers
        preds = np.argmax(scaled_probs, axis=1)

        batch_start = i * BATCH_SIZE
        batch_end = batch_start + images.size(0)
        file_names.extend(test_dataset.annotations.iloc[batch_start:batch_end]["file_name"].values)
        
        # Chuyển đổi nhãn từ 0-5 về 1-6 chuẩn quy cách BTC
        predictions.extend(preds + 1)

# 5. Tạo DataFrame và lưu file submission.csv
submission_df = pd.DataFrame({
    "file_name": file_names,
    "category_id": predictions
})

submission_path = os.path.join(ROOT_DIR, "submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"\n=== ĐÃ TẠO FILE NỘP BÀI THÀNH CÔNG ===")
print(f"Đường dẫn: {submission_path}")
print(f"Tổng số bản ghi: {len(submission_df)}")
print("\n5 dòng đầu tiên:")
print(submission_df.head(5))
print("\nPhân phối nhãn dự đoán:")
print(submission_df['category_id'].value_counts().sort_index())


In [ ]:
# 9. Kiểm tra tính hợp lệ của file submission.csv theo tiêu chí BTC
def verify_submission(csv_path):
    assert os.path.exists(csv_path), f"File không tồn tại: {csv_path}"
    df = pd.read_csv(csv_path)
    
    # Kiểm tra cột
    assert list(df.columns) == ["file_name", "category_id"], f"Tên cột không hợp lệ: {list(df.columns)}"
    
    # Kiểm tra giá trị rỗng
    assert df.isnull().sum().sum() == 0, "File chứa giá trị rỗng NaN!"
    
    # Kiểm tra miền giá trị category_id từ 1 đến 6
    unique_labels = sorted(df['category_id'].unique())
    assert all(l in [1, 2, 3, 4, 5, 6] for l in unique_labels), f"Nhãn vượt ngoài [1..6]: {unique_labels}"
    
    print(f"File {csv_path} HOÀN TOÀN HỢP LỆ theo quy định BTC:")
    print(f"- Số dòng dự đoán: {len(df)}")
    print(f"- Cột: {list(df.columns)}")
    print(f"- Các nhãn xuất hiện: {unique_labels}")

verify_submission(submission_path)
